In [1]:
import pandas as pd
import xarray as xr

In [26]:
ds = xr.open_dataset("/Volumes/T7/Data/EOBS/tx_ens_mean_0.1deg_reg_v31.0e.nc")
ds

<xarray.Dataset> Size: 36GB
Dimensions:    (longitude: 705, latitude: 465, time: 27394)
Coordinates:
  * longitude  (longitude) float64 6kB -24.95 -24.85 -24.75 ... 45.35 45.45
  * latitude   (latitude) float64 4kB 25.05 25.15 25.25 ... 71.25 71.35 71.45
  * time       (time) datetime64[ns] 219kB 1950-01-01 1950-01-02 ... 2024-12-31
Data variables:
    tx         (time, latitude, longitude) float32 36GB ...
Attributes:
    E-OBS_version:  31.0e
    Conventions:    CF-1.4
    References:     http://surfobs.climate.copernicus.eu/dataaccess/access_eo...
    history:        Mon Mar  3 13:14:36 2025: ncks --no-abc -d time,0,27393 /...
    NCO:            netCDF Operators version 5.2.2 (Homepage = http://nco.sf....

In [27]:
ds_sel = ds.sel(time=slice("1960-01-01", None))

In [28]:
def extract_eobs_timeseries(ds, var_name, lat, lon, start=None, end=None):
    """
    Extract a time series from an E-OBS dataset at a given location
    and within a selected time period.

    Parameters
    ----------
    ds : xarray.Dataset
        The loaded E-OBS dataset.
    var_name : str
        Name of the variable to extract, e.g. "tg".
    lat : float
        Latitude of the desired location.
    lon : float
        Longitude of the desired location.
    start : str or None
        Start date in format 'YYYY-MM-DD' (e.g. '1960-01-01').
        If None, no lower time bound is applied.
    end : str or None
        End date in format 'YYYY-MM-DD'.
        If None, no upper time bound is applied.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing time, lat, lon, and the variable values.
    """

    # Select time range if provided
    if start or end:
        ds = ds.sel(time=slice(start, end))

    # Extract nearest grid point for the variable
    ts = ds[var_name].sel(latitude=lat, longitude=lon, method="nearest")

    # Convert to DataFrame
    df = ts.to_dataframe().reset_index()

    return df

In [29]:
def get_minimum_necessary_df(large_df, variable):
    df = pd.DataFrame()

    df["time"] = large_df["time"]
    df[variable] = large_df[variable]

    return df

In [30]:
locations = {
    "Cluj": (46.75, 23.65),
    "Gheorgheni": (46.75, 25.45),
    "Brasov": (45.75, 25.55),
    "Deva": (45.85, 22.95),
    "Pecs": (46.05, 18.25),
    "Gyor": (47.75, 17.65),
    "Oradea": (47.15, 21.85),
    "Kassa": (48.65, 21.25),
    "Kecskemet": (46.85, 19.75),
    "Keszthely": (46.75, 17.15),
    "Bacskatopolya": (45.85, 19.65),
}

In [31]:
current_variable = "tx"

for key in locations:

    print(f"Working with {key} : {locations[key]}...")

    df_tn_loc = extract_eobs_timeseries(
        ds,
        var_name=current_variable,
        lat=locations[key][0],
        lon=locations[key][1],
        start="1960-01-01",
        end=None
    )

    print(f"Got large df")

    df_min_loc = get_minimum_necessary_df(df_tn_loc, current_variable)

    print(f"Small df is ready")

    df_min_loc.to_csv(f"../{current_variable}/{key}_{current_variable}.csv", index = False)

    print(f"Data saved to ../{current_variable}/{key}_{current_variable}.csv")

    print(f"\n\n\n")

Working with Cluj : (46.75, 23.65)...
Got large df
Small df is ready
Data saved to ../tx/Cluj_tx.csv




Working with Gheorgheni : (46.75, 25.45)...
Got large df
Small df is ready
Data saved to ../tx/Gheorgheni_tx.csv




Working with Brasov : (45.75, 25.55)...
Got large df
Small df is ready
Data saved to ../tx/Brasov_tx.csv




Working with Deva : (45.85, 22.95)...
Got large df
Small df is ready
Data saved to ../tx/Deva_tx.csv




Working with Pecs : (46.05, 18.25)...
Got large df
Small df is ready
Data saved to ../tx/Pecs_tx.csv




Working with Gyor : (47.75, 17.65)...
Got large df
Small df is ready
Data saved to ../tx/Gyor_tx.csv




Working with Oradea : (47.15, 21.85)...
Got large df
Small df is ready
Data saved to ../tx/Oradea_tx.csv




Working with Kassa : (48.65, 21.25)...
Got large df
Small df is ready
Data saved to ../tx/Kassa_tx.csv




Working with Kecskemet : (46.85, 19.75)...
Got large df
Small df is ready
Data saved to ../tx/Kecskemet_tx.csv




Working with Keszthely 